In [1]:
import requests
import pandas as pd
import time

##### Fetch all documents for the specified session

In [2]:
def get_documents(session_number: int):
    """
    Obtain all documents for the specified session of the General Court
    """
    # Returns all of documents publicly available by the General Court
    url = f"https://malegislature.gov/api/GeneralCourts/{session_number}/Documents/"
    print(f"\n Fetching documents from General Court {session_number} ...")

    response = requests.get(url)
    response.raise_for_status()
    docs = response.json()
    print(f"Successfully retrieved {len(docs)} documents from General Court {session_number}.")
    
    return docs


In [ ]:
#docs_193 = get_documents(193)

#for i, doc in enumerate(docs_193[:3]):
#   print(f"\n--- Document {i+1} preview ---")
#    print(doc)



 Fetching documents from General Court 193 ...
Successfully retrieved 10156 documents from General Court 193.


##### Extract information from documents

In [ ]:
import pandas as pd

def extract_bill_info(docs_json, session_number):
    """
    Extract key information from the documents JSON of the Legislature API 
    and convert it into DataFrame format.
    
    Return:
        pd.DataFrame: 
        containing billnumbers, docketnumbers, title, branch, and IsDocketBookOnly
    """
    bill_list = []

    for doc in docs_json:
        bill_no = doc.get("BillNumber")
        docket_no = doc.get("DocketNumber")
        title = doc.get("Title")
        is_book_only = doc.get("IsDocketBookOnly")

        bill_list.append({
            "billNumber": bill_no if bill_no else None,
            "docketNumber": docket_no if docket_no else None,
            "title": title if title else None,
            "branch": (
                "Senate" if bill_no and bill_no.startswith("S") else
                "House" if bill_no and bill_no.startswith("H") else None
            ),
            "isDocketBookOnly": is_book_only
        })

    df = pd.DataFrame(bill_list)
    print(f"Extracted {len(df)} entries from documents for General Court {session_number if session_number else ''}...")
    #print("Preview of first 5 rows:")
    #print(df.head())
    
    print("\nMissing value summary:")
    print(df.isna().sum())

    print("\n Count by branch:")
    print(df['branch'].value_counts(dropna=False))

    return df


#df_193 = extract_bill_info(docs_193, 193)


Extracted 10156 entries from documents for General Court 193...

Missing value summary:
billNumber          1973
docketNumber        1062
title                  0
branch              1973
isDocketBookOnly       0
dtype: int64

 Count by branch:
branch
House     5165
Senate    3018
None      1973
Name: count, dtype: int64


##### Filter bills containing rollcall records

In [22]:

def get_rollcall_status(df_bills, session_number=None):
    """
    Scan each bill to determine whether a roll call vote record exists.
    """

    # Filter bills with billnumbers：billnumber is not null + Senate bills（the API does not return House records)
    bill_numbers = df_bills[(df_bills['billNumber'].notna()) &
                            (df_bills['branch'] == 'Senate')]['billNumber'].tolist()
    total = len(bill_numbers)
    rollcall_records = []

    print(f"\nStarting roll call scan for General Court {session_number if session_number else ''}")
    print(f"Total bills to scan: {total}\n")

    for index, bill in enumerate(bill_numbers):
        url = f"https://malegislature.gov/api/GeneralCourts/{session_number}/Documents/{bill}/RollCalls"
        

        try:
            resp = requests.get(url, timeout=10)
            resp.raise_for_status()
            data = resp.json()

            # Extract roll call number
            if isinstance(data, list) and len(data) > 0:
                roll_nums = [item.get("RollCallNumber") for item in data if "RollCallNumber" in item]
            else:
                roll_nums = []

            rollcall_records.append({
                "billNumber": bill,
                "rollCallNumbers": roll_nums,
                "hasRollCall": len(roll_nums) > 0
            })

        except Exception as e:
            rollcall_records.append({
                "billNumber": bill,
                "rollCallNumbers": None,
                "hasRollCall": None
            })
            #print(f"Error fetching {bill}: {e}")

        if (index + 1) % 500 == 0 or (index + 1) == total:
            print(f"📍 Progress: {index + 1}/{total} bills processed...")

        #time.sleep(delay)

    df_rollcall = pd.DataFrame(rollcall_records)

    print("\nRoll Call Scan Summary:")
    print(df_rollcall['hasRollCall'].value_counts(dropna=False))
    print("\nExample preview:")
    print(df_rollcall.head())
    print(df_rollcall.shape)

    return df_rollcall

#df_bills_193 = get_rollcall_status(df_193, session_number=193)


##### Fetch the details of each vote by every member

In [5]:
def fetch_bill_vote_details(df_roll_status, session_number=None, delay=0.2):
    """
    Retrieve all voting details for legislators who participated in roll call votes 
    from the existing bill roll call status table.
    """
    # Select only bills that have had roll call votes.
    df_with_rolls = df_roll_status[df_roll_status['hasRollCall'] == True].copy()
    total_bills = len(df_with_rolls)
    print(f"\nFetching vote details for {total_bills} bills with roll calls "
          f"in General Court {session_number if session_number else ''}...\n")

    all_vote_records = []

    # Iterate through each bill with a roll call
    for idx, row in df_with_rolls.iterrows():
        bill_no = row['billNumber']
        roll_calls = row['rollCallNumbers'] if isinstance(row['rollCallNumbers'], list) else []
        if not roll_calls:
            continue

        for roll_num in roll_calls:
            url = f"https://malegislature.gov/api/GeneralCourts/{session_number}/Branches/Senate/RollCalls/{roll_num}"
            try:
                resp = requests.get(url, timeout=8)
                resp.raise_for_status()
                data = resp.json()

                # Extract the list of votes
                yea_list = data.get("Yeas", [])
                nay_list = data.get("Nays", [])
                absent_list = data.get("Absent", [])

                for vote_list, label in [(yea_list, "Yea"), (nay_list, "Nay"), (absent_list, "Absent")]:
                    for member in vote_list:
                        all_vote_records.append({
                            "GeneralCourtNumber": data.get("GeneralCourtNumber"),
                            "Branch": data.get("Branch"),
                            "RollCallNumber": roll_num,
                            "QuestionMotion": data.get("QuestionMotion"),  
                            "BillNumber": bill_no,
                            "MemberCode": member.get("MemberCode"),
                            "Vote": label,
                            "MemberUrl": member.get("Details")
                        })

                #print(f"RollCall {roll_num} from {bill_no}: "
                      #f"{len(yea_list)}Y / {len(nay_list)}N / {len(absent_list)}A")

            except Exception as e:
                print(f"Error fetching {bill_no} - RollCall {roll_num}: {e}")

            time.sleep(delay)  

    df_votes = pd.DataFrame(all_vote_records)
    print(f"\n Completed. Total vote records fetched: {len(df_votes)}")
    return df_votes


#df_vote_details_193 = fetch_bill_vote_details(df_bills_193, session_number=193, delay=0.1)

In [7]:
#df_vote_details_193.head()

In [8]:
#df_with_rolls = df_bills_193[df_bills_193['hasRollCall'] == True].copy()
#df_with_rolls

##### Fetch legislators info for the specified session

In [6]:
def fetch_legislators_from_votes(df_vote_details, session_number, delay=0.2):
    """
    Extract all MemberCodes from the vote details table,
    and call the API to retrieve detailed information on all legislators for that session.
    """

    # Extract all unique MemberCodes
    unique_codes = df_vote_details["MemberCode"].dropna().unique().tolist()
    total = len(unique_codes)
    print(f"\nFound {total} unique legislators in General Court {session_number}.\n")

    records = []

    # Iterate through each legislator and request data
    for idx, code in enumerate(unique_codes, start=1):
        url = f"https://malegislature.gov/api/GeneralCourts/{session_number}/LegislativeMembers/{code}"
        try:
            resp = requests.get(url, timeout=12)
            resp.raise_for_status()
            data = resp.json()

            record = {
                "MemberCode": data.get("MemberCode"),
                "Name": data.get("Name"),
                "Branch": data.get("Branch"),
                "Party": data.get("Party"),
                "District": data.get("District"),
                "LeadershipPosition": data.get("LeadershipPosition"),
                "EmailAddress": data.get("EmailAddress"),
                "PhoneNumber": data.get("PhoneNumber"),
                "RoomNumber": data.get("RoomNumber"),
            }
            records.append(record)
            #print(f"{idx}/{total} {code} — {record['Name']}")

        except Exception as e:
            print(f"Error fetching {code}: {e}")
            continue

        time.sleep(delay)

    df_legislators = pd.DataFrame(records)
    print(f"\nRetrieved {len(df_legislators)} legislators for General Court {session_number}.")
    
    return df_legislators


In [9]:
#df_legislators_193 = fetch_legislators_from_votes(df_vote_details_193, session_number=193, delay=1)

#df_legislators_193.head()


In [23]:
def build_session_data(session_number, delay_rollcall=0.1, delay_vote=0.1, delay_legislator=0.2):
    """
    Enter the General Court session number and execute the complete data extraction process step by step.
    Return the voting details table and member information table for that session.
    """

    print(f"\nBuilding data for General Court {session_number} ...")

    # Step 1: extract documents
    docs_json = get_documents(session_number)
    df_bills = extract_bill_info(docs_json, session_number)

    # Step 2: extract roll call status
    df_roll_status = get_rollcall_status(df_bills, session_number=session_number)

    # Step 3: extract roll call vote details
    df_vote_details = fetch_bill_vote_details(
        df_roll_status,
        session_number=session_number,
        delay=delay_vote
    )

    # Step 4: extract legislators info
    df_legislators = fetch_legislators_from_votes(
        df_vote_details,
        session_number=session_number,
        delay=delay_legislator
    )

    print(f"Completed session {session_number}.")
    print(f"   Votes: {len(df_vote_details)} records, Legislators: {len(df_legislators)} entries.\n")

    return df_vote_details, df_legislators


In [24]:
df_vote_details_193, df_legislators_193 = build_session_data(session_number=193)



Building data for General Court 193 ...

 Fetching documents from General Court 193 ...
Successfully retrieved 10156 documents from General Court 193.
Extracted 10156 entries from documents for General Court 193...
Preview of first 5 rows:
  billNumber docketNumber                                              title  \
0      H2105         HD96  An Act authorizing municipal use of the pruden...   
1       H563         HD80  An Act concerning disposable menstrual product...   
2      H1751       HD1786  An Act concerning payments on personal injury ...   
3      H2395         HD85        An Act establishing a primary seat belt law   
4       H421        HD250  An Act protecting the Commonwealth’s gaming in...   

  branch  isDocketBookOnly  
0  House             False  
1  House             False  
2  House             False  
3  House             False  
4  House             False  

Missing value summary:
billNumber          1973
docketNumber        1062
title                  0
branc

In [37]:
df_vote_details_192, df_legislators_192 = build_session_data(session_number=192)


Building data for General Court 192 ...

 Fetching documents from General Court 192 ...
Successfully retrieved 10363 documents from General Court 192.
Extracted 10363 entries from documents for General Court 192...

Missing value summary:
billNumber          1718
docketNumber        1295
title                  0
branch              1720
isDocketBookOnly       0
dtype: int64

 Count by branch:
branch
House     5462
Senate    3181
None      1720
Name: count, dtype: int64

Starting roll call scan for General Court 192
Total bills to scan: 3181

📍 Progress: 500/3181 bills processed...
📍 Progress: 1000/3181 bills processed...
📍 Progress: 1500/3181 bills processed...
📍 Progress: 2000/3181 bills processed...
📍 Progress: 2500/3181 bills processed...
📍 Progress: 3000/3181 bills processed...
📍 Progress: 3181/3181 bills processed...

Roll Call Scan Summary:
hasRollCall
None    3121
True      60
Name: count, dtype: int64

Example preview:
  billNumber rollCallNumbers hasRollCall
0      S1160     

In [43]:
#df_legislators_192["Name"].unique()

In [53]:
df_vote_details_191, df_legislators_191 = build_session_data(session_number=191)


Building data for General Court 191 ...

 Fetching documents from General Court 191 ...
Successfully retrieved 9690 documents from General Court 191.
Extracted 9690 entries from documents for General Court 191...

Missing value summary:
billNumber          1437
docketNumber        1157
title                  0
branch              1437
isDocketBookOnly       0
dtype: int64

 Count by branch:
branch
House     5248
Senate    3005
None      1437
Name: count, dtype: int64

Starting roll call scan for General Court 191
Total bills to scan: 3005

📍 Progress: 500/3005 bills processed...
📍 Progress: 1000/3005 bills processed...
📍 Progress: 1500/3005 bills processed...
📍 Progress: 2000/3005 bills processed...
📍 Progress: 2500/3005 bills processed...
📍 Progress: 3000/3005 bills processed...
📍 Progress: 3005/3005 bills processed...

Roll Call Scan Summary:
hasRollCall
None    2938
True      67
Name: count, dtype: int64

Example preview:
  billNumber rollCallNumbers hasRollCall
0      S2065       

##### Merge datasets


In [30]:
def merge_legislator_votes(vote_file, legislator_file, output_file=None):
    """
    Merge the legislator info sheet with the voting details sheet into a single comprehensive dataset.
    """
    if isinstance(vote_file, str):
        votes = pd.read_csv(vote_file)
    else:
        votes = vote_file.copy()

    if isinstance(legislator_file, str):
        members = pd.read_csv(legislator_file)
    else:
        members = legislator_file.copy()

    print(f"Loaded {len(votes)} vote records and {len(members)} legislators.")

    # Remove duplicate branch to avoid conflicts
    votes = votes.drop(columns=["Branch"], errors="ignore")

    merged = pd.merge(
        members,
        votes,
        on="MemberCode",
        how="left"  
    )

    columns = [
        "GeneralCourtNumber", "MemberCode", "Name", "Branch", 
        "BillNumber", "RollCallNumber", "QuestionMotion", "Vote", "Party", "District",
        "LeadershipPosition", "EmailAddress", "PhoneNumber", "RoomNumber", "MemberUrl"
    ]
    merged = merged[[c for c in columns if c in merged.columns]]

    # Sorted by Member and Voting Order
    merged["order"] = merged.index
    merged = merged.sort_values(by=["order", "RollCallNumber"], ascending=[True, True])
    merged = merged.drop(columns="order")

    print(f"Merged {len(merged)} rows.")
    return merged



In [35]:
merged_df = merge_legislator_votes(df_vote_details_193, df_legislators_193)
merged_df.head()

Loaded 6242 vote records and 41 legislators.
Merged 6242 rows.


,GeneralCourtNumber,MemberCode,Name,Branch,BillNumber,RollCallNumber,QuestionMotion,Vote,Party,District,LeadershipPosition,EmailAddress,PhoneNumber,RoomNumber,MemberUrl
0,193,PWM0,Paul W. Mark,Senate,S2251,58,None,Yea,Democrat,"Berkshire, Hampden, Franklin and Hampshire",None,Paul.Mark@masenate.gov,6177221625,504,http://malegislature.gov/api/GeneralCourts/193...
1,193,PWM0,Paul W. Mark,Senate,S17,2,Question on adoption of the amendment 1,Nay,Democrat,"Berkshire, Hampden, Franklin and Hampshire",None,Paul.Mark@masenate.gov,6177221625,504,http://malegislature.gov/api/GeneralCourts/193...
2,193,PWM0,Paul W. Mark,Senate,S17,3,Question on adoption of the amendment 2,Nay,Democrat,"Berkshire, Hampden, Franklin and Hampshire",None,Paul.Mark@masenate.gov,6177221625,504,http://malegislature.gov/api/GeneralCourts/193...
3,193,PWM0,Paul W. Mark,Senate,S17,4,Question on adoption of the amendment 5,Nay,Democrat,"Berkshire, Hampden, Franklin and Hampshire",None,Paul.Mark@masenate.gov,6177221625,504,http://malegislature.gov/api/GeneralCourts/193...
4,193,PWM0,Paul W. Mark,Senate,S17,5,Question on adoption of the amendment 6,Nay,Democrat,"Berkshire, Hampden, Franklin and Hampshire",None,Paul.Mark@masenate.gov,6177221625,504,http://malegislature.gov/api/GeneralCourts/193...


In [34]:
merged_df.to_csv("193_merged_legislator_votes.csv", index=False)


In [32]:
df_vote_details_193.head()

,GeneralCourtNumber,Branch,RollCallNumber,QuestionMotion,BillNumber,MemberCode,Vote,MemberUrl
0,193,Senate,58,None,S2251,PWM0,Yea,http://malegislature.gov/api/GeneralCourts/193...
1,193,Senate,58,None,S2251,PMP0,Yea,http://malegislature.gov/api/GeneralCourts/193...
2,193,Senate,58,None,S2251,RKK0,Yea,http://malegislature.gov/api/GeneralCourts/193...
3,193,Senate,58,None,S2251,JRO0,Yea,http://malegislature.gov/api/GeneralCourts/193...
4,193,Senate,58,None,S2251,L M0,Yea,http://malegislature.gov/api/GeneralCourts/193...


In [33]:
print(df_vote_details_193.columns.tolist())
print(df_legislators_193.columns.tolist())


['GeneralCourtNumber', 'Branch', 'RollCallNumber', 'QuestionMotion', 'BillNumber', 'MemberCode', 'Vote', 'MemberUrl']
['MemberCode', 'Name', 'Branch', 'Party', 'District', 'LeadershipPosition', 'EmailAddress', 'PhoneNumber', 'RoomNumber']
